# A small search, then a comparison

`tune_forest` tries 8 forests on the train rows, using 3-fold ROC-AUC. The grid is short on purpose: 40, 80, or 120 trees, depth 3, 5, or 8, and at least 5, 10, or 20 rows in a leaf.

If this cell is still running at 14:20, stop it and call `tune_forest` with `n_iter=4`.

Majority ROC-AUC is 0.5 because a constant score does not rank a churner above a stayer. Accuracy can look fine while the ranking is useless. Compare AUC.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = here
if not (root / "day1").is_dir() and (root.parent / "day1").is_dir():
    root = root.parent
if not (root / "day1").is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from the notebooks folder."
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
ROOT = root
print("cwd", here)
print("repo", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "klongcart_churn.csv")
print(customers.shape)
customers.head()


In [ ]:
from day1.frame import features_and_target
from day1.split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.metrics import roc_auc_score

from day1.models import make_forest_pipeline, make_logistic_pipeline, tune_forest

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
forest = make_forest_pipeline()
forest.fit(X_train, y_train)
search = tune_forest(X_train, y_train, n_iter=8)
tuned = search.best_estimator_
print(search.best_params_)


def auc(model, features, labels):
    return float(roc_auc_score(labels, model.predict_proba(features)[:, 1]))


tuned_train_auc = auc(tuned, X_train, y_train)
tuned_val_auc = auc(tuned, X_val, y_val)
logistic_val_auc = auc(logistic, X_val, y_val)
forest_val_auc = auc(forest, X_val, y_val)
print("logistic validation", round(logistic_val_auc, 3))
print("forest validation", round(forest_val_auc, 3))
print("tuned train", round(tuned_train_auc, 3))
print("tuned validation", round(tuned_val_auc, 3))


### Exercises

`beats_baseline` is True when the tuned validation AUC is above 0.5.

`gap` is the tuned train AUC minus the tuned validation AUC. A large gap means the train score is flattering. Write one sentence under the code about whether you would show this model to the team.


In [ ]:
beats_baseline = None
gap = None


In [ ]:
if beats_baseline is None or gap is None:
    raise NotImplementedError("Finish the baseline and gap exercises.")
print("beats majority AUC", bool(beats_baseline))
print("train minus validation", round(float(gap), 3))


Write one sentence here. Is the tuned validation score above 0.5, and is the gap small enough to keep?


## Checkpoint

Notebook 05 fits these models again from the CSV, so you do not need to keep this kernel. Do not look at the test rows in this file.
